# Aria Localization & Internationalization (i18n) Guide

**Complete guide for multi-language support, regional adaptations, and global user experiences.**

Learn how to localize Aria for users worldwide.


## Localization Architecture

### Multi-Language Strategy

```
Aria Platform
├─ Backend APIs (language-agnostic)
├─ Language Detection
│  ├─ User preference
│  ├─ Browser language
│  └─ Geo-location
├─ Translation Service
│  ├─ Static strings (UI)
│  ├─ Dynamic content (responses)
│  └─ User-generated content
└─ Regional Adaptations
   ├─ Date/time formats
   ├─ Currency
   ├─ Number formats
   └─ Cultural sensitivities
```

### Supported Languages

```python
# config/i18n_config.yaml
supported_languages:
  tier_1:  # Production-ready
    - en-US
    - es-ES
    - fr-FR
    - de-DE
    - ja-JP
    - zh-CN

  tier_2:  # Community translations
    - pt-BR
    - ru-RU
    - ko-KR
    - ar-SA
    - hi-IN

language_detection_priority:
  - user_preference  # Highest priority
  - browser_language
  - geo_location
  - default: en-US

character_limits:
  en-US: 100
  de-DE: 120  # German tends to be longer
  ja-JP: 30   # Japanese uses fewer characters
```

### Translation Management

```python
# shared/i18n.py
import json
from typing import Dict, Optional

class I18nManager:
    def __init__(self):
        self.translations = self._load_translations()
        self.default_language = "en-US"

    def _load_translations(self) -> Dict:
        """Load all translation files."""
        translations = {}
        import os
        translation_dir = "locales"

        for filename in os.listdir(translation_dir):
            if filename.endswith(".json"):
                lang = filename.replace(".json", "")
                with open(f"{translation_dir}/{filename}") as f:
                    translations[lang] = json.load(f)

        return translations

    def translate(self, key: str, language: str = None, **kwargs) -> str:
        """Get translated string with optional interpolation."""
        lang = language or self.default_language

        # Fall back to English if translation not found
        if lang not in self.translations:
            lang = self.default_language

        translation_tree = self.translations[lang]

        # Navigate nested keys (e.g., "chat.welcome.title")
        for part in key.split("."):
            if isinstance(translation_tree, dict):
                translation_tree = translation_tree.get(part, key)
            else:
                return key

        # Interpolate variables
        if kwargs:
            return translation_tree.format(**kwargs)

        return translation_tree

    def translate_list(self, keys: list, language: str = None) -> list:
        """Translate multiple strings."""
        return [self.translate(key, language) for key in keys]

# Global instance
i18n = I18nManager()

# Usage in code
def get_welcome_message(user_name: str, language: str = "en-US") -> str:
    return i18n.translate(
        "chat.welcome.title",
        language=language,
        name=user_name
    )
```

### Translation File Format

```json
{
    "chat": {
        "welcome": {
            "title": "Hello, {name}!",
            "subtitle": "How can I help you today?",
            "placeholder": "Type your message..."
        },
        "errors": {
            "timeout": "Request timed out. Please try again.",
            "invalid_input": "Invalid input. Please check and retry.",
            "rate_limit": "Too many requests. Wait a moment."
        }
    },
    "aria": {
        "actions": {
            "move": "Move",
            "wave": "Wave",
            "dance": "Dance"
        }
    },
    "common": {
        "loading": "Loading...",
        "error": "An error occurred",
        "retry": "Retry"
    }
}
```

---

## Frontend Localization

### React i18n Implementation

```typescript
// apps/chat/src/i18n/useTranslation.ts
import { useContext, useMemo } from 'react';
import { I18nContext } from './I18nContext';

export function useTranslation() {
  const { language, translations } = useContext(I18nContext);

  const t = useMemo(() => (key: string, params?: Record<string, any>) => {
    let value = translations[language]?.[key] || key;

    // Replace parameters
    if (params) {
      Object.entries(params).forEach(([k, v]) => {
        value = value.replace(`{${k}}`, String(v));
      });
    }

    return value;
  }, [language, translations]);

  return { t, language };
}

// Usage in components
import { useTranslation } from './i18n/useTranslation';

export function ChatWelcome() {
  const { t } = useTranslation();

  return (
    <div>
      <h1>{t('chat.welcome.title', { name: 'User' })}</h1>
      <p>{t('chat.welcome.subtitle')}</p>
      <input placeholder={t('chat.welcome.placeholder')} />
    </div>
  );
}
```

### Language Selector Component

```typescript
// apps/chat/src/components/LanguageSelector.tsx
import { useContext } from 'react';
import { I18nContext } from '../i18n/I18nContext';

export function LanguageSelector() {
  const { language, setLanguage } = useContext(I18nContext);

  const languages = [
    { code: 'en-US', name: 'English' },
    { code: 'es-ES', name: 'Español' },
    { code: 'fr-FR', name: 'Français' },
    { code: 'de-DE', name: 'Deutsch' },
    { code: 'ja-JP', name: '日本語' },
    { code: 'zh-CN', name: '中文' }
  ];

  return (
    <select value={language} onChange={(e) => setLanguage(e.target.value)}>
      {languages.map(lang => (
        <option key={lang.code} value={lang.code}>
          {lang.name}
        </option>
      ))}
    </select>
  );
}
```

---

## Regional Adaptations

### Date & Time Formatting

```python
# shared/localization.py
from datetime import datetime
import locale

class RegionalFormatter:
    """Format dates, times, and numbers by region."""

    LOCALE_MAP = {
        "en-US": "en_US.UTF-8",
        "es-ES": "es_ES.UTF-8",
        "fr-FR": "fr_FR.UTF-8",
        "de-DE": "de_DE.UTF-8",
        "ja-JP": "ja_JP.UTF-8",
        "zh-CN": "zh_CN.UTF-8"
    }

    def __init__(self, language_code: str):
        self.language_code = language_code
        self.locale_name = self.LOCALE_MAP.get(language_code, "en_US.UTF-8")

    def format_date(self, dt: datetime) -> str:
        """Format date in regional style."""
        try:
            locale.setlocale(locale.LC_TIME, self.locale_name)
        except locale.Error:
            pass  # Fall back to system locale

        if self.language_code == "en-US":
            return dt.strftime("%m/%d/%Y")  # 07/05/2026
        elif self.language_code == "de-DE":
            return dt.strftime("%d.%m.%Y")  # 05.07.2026
        elif self.language_code == "ja-JP":
            return dt.strftime("%Y年%m月%d日")  # 2026年7月5日
        else:
            return dt.strftime("%d/%m/%Y")  # 05/07/2026

    def format_time(self, dt: datetime, include_seconds: bool = False) -> str:
        """Format time in regional style."""
        if self.language_code in ["en-US"]:
            fmt = "%I:%M %p" if not include_seconds else "%I:%M:%S %p"
            return dt.strftime(fmt)  # 2:30 PM or 2:30:45 PM
        else:
            fmt = "%H:%M" if not include_seconds else "%H:%M:%S"
            return dt.strftime(fmt)  # 14:30 or 14:30:45

    def format_number(self, number: float, decimal_places: int = 2) -> str:
        """Format number with regional separators."""
        if self.language_code in ["en-US", "ja-JP"]:
            return f"{number:,.{decimal_places}f}"  # 1,234.56
        else:
            # European style
            return f"{number:,.{decimal_places}f}".replace(",", "X").replace(".", ",").replace("X", ".")

    def format_currency(self, amount: float, currency_code: str = "USD") -> str:
        """Format currency with regional style."""
        symbol_map = {
            "USD": "$",
            "EUR": "€",
            "GBP": "£",
            "JPY": "¥",
            "CNY": "¥"
        }

        symbol = symbol_map.get(currency_code, currency_code)
        formatted_number = self.format_number(amount, 2)

        if self.language_code in ["en-US"]:
            return f"{symbol}{formatted_number}"  # $1,234.56
        elif self.language_code in ["de-DE", "fr-FR"]:
            return f"{formatted_number} {symbol}"  # 1.234,56 €
        else:
            return f"{symbol}{formatted_number}"

# Usage
formatter_us = RegionalFormatter("en-US")
formatter_de = RegionalFormatter("de-DE")

now = datetime.now()
print(formatter_us.format_date(now))  # 07/05/2026
print(formatter_de.format_date(now))  # 05.07.2026
```

### Cultural Sensitivity

```python
# Avoid culture-specific references
class CulturalContent:
    """Manage culturally-sensitive content."""

    AVOID_IN_REGIONS = {
        "hand_gestures": {
            "thumbs_up": ["middle_east"],  # Offensive in some cultures
            "ok_sign": ["france"],  # Means something different
            "pointing": ["philippines"]
        },
        "colors": {
            "red": ["eastern_cultures"],  # Mourning color
            "white": ["east_asia"],  # Death color
            "black": ["western_cultures"]  # Mourning color
        },
        "symbols": {
            "owl": ["asian_cultures"],  # Bad luck
            "clock": ["china"]  # Similar to "funeral"
        }
    }

    def is_safe_for_region(self, content_type: str, content_value: str, region: str) -> bool:
        """Check if content is culturally appropriate."""
        unsafe = self.AVOID_IN_REGIONS.get(content_type, {})
        restricted_regions = unsafe.get(content_value, [])
        return region not in restricted_regions
```


## Translation Workflow

### Adding a New Language

```bash
#!/bin/bash
# scripts/add_language.sh

LANGUAGE=$1
LANGUAGE_CODE=$2

if [ -z "$LANGUAGE" ] || [ -z "$LANGUAGE_CODE" ]; then
    echo "Usage: ./add_language.sh <Language> <language-code>"
    echo "Example: ./add_language.sh German de-DE"
    exit 1
fi

# Create translation file
cat > locales/${LANGUAGE_CODE}.json << 'EOF'
{
  "chat": {
    "welcome": {
      "title": "Translate: Hello, {name}!",
      "subtitle": "Translate: How can I help you today?"
    }
  }
}
EOF

# Update config
echo "  - $LANGUAGE_CODE  # $LANGUAGE" >> config/i18n_config.yaml

# Create translation guide
cat > docs/TRANSLATE_${LANGUAGE_CODE}.md << 'EOF'
# Translation Guide for $LANGUAGE ($LANGUAGE_CODE)

## Instructions
1. Open `locales/${LANGUAGE_CODE}.json`
2. Replace all English strings with $LANGUAGE translations
3. Maintain JSON structure
4. Test with: `python scripts/test_translations.py --lang ${LANGUAGE_CODE}`
5. Submit PR for review

## Common Phrases
- "Hello" →
- "Error" →
- "Loading" →

## Notes
- Maintain string length constraints
- Check character direction (RTL for Arabic)
- Verify emoji support
EOF

echo "✓ Language ${LANGUAGE_CODE} added"
echo "✓ Translation guide: docs/TRANSLATE_${LANGUAGE_CODE}.md"
```

### Translation Quality Assurance

```python
# scripts/test_translations.py
import json
from pathlib import Path

class TranslationValidator:
    def __init__(self, language_code: str):
        self.language_code = language_code
        self.base_translations = self._load_base()
        self.target_translations = self._load_target()

    def _load_base(self):
        """Load English translations as reference."""
        with open("locales/en-US.json") as f:
            return json.load(f)

    def _load_target(self):
        """Load target language translations."""
        with open(f"locales/{self.language_code}.json") as f:
            return json.load(f)

    def validate(self) -> list:
        """Check for common translation issues."""
        issues = []

        # Check 1: All keys present
        if self._get_keys(self.base_translations) != self._get_keys(self.target_translations):
            issues.append("Missing or extra keys")

        # Check 2: Parameter placeholders preserved
        for key in self._flatten_keys(self.base_translations):
            base = self._get_nested(self.base_translations, key)
            target = self._get_nested(self.target_translations, key)

            base_params = self._extract_params(base)
            target_params = self._extract_params(target)

            if base_params != target_params:
                issues.append(f"Parameter mismatch in {key}: {base_params} vs {target_params}")

        # Check 3: String length reasonableness
        for key in self._flatten_keys(self.target_translations):
            base = self._get_nested(self.base_translations, key)
            target = self._get_nested(self.target_translations, key)

            if len(target) > len(base) * 1.5:
                issues.append(f"String too long in {key}: {len(target)} chars (base: {len(base)})")

        return issues

    def _get_keys(self, d):
        """Get all nested keys."""
        keys = set()
        for k, v in d.items():
            if isinstance(v, dict):
                for nested_key in self._get_keys(v):
                    keys.add(f"{k}.{nested_key}")
            else:
                keys.add(k)
        return keys

    def _extract_params(self, text: str) -> set:
        """Extract {param} style placeholders."""
        import re
        return set(re.findall(r'\{(\w+)\}', text))

    def _flatten_keys(self, d, prefix=""):
        """Flatten nested dict to keys."""
        for k, v in d.items():
            key = f"{prefix}.{k}" if prefix else k
            if isinstance(v, dict):
                yield from self._flatten_keys(v, key)
            else:
                yield key

    def _get_nested(self, d, key):
        """Get value from nested dict using dot notation."""
        for part in key.split("."):
            d = d[part]
        return d

# Run validation
if __name__ == "__main__":
    import sys
    lang = sys.argv[1] if len(sys.argv) > 1 else "es-ES"
    validator = TranslationValidator(lang)
    issues = validator.validate()

    if issues:
        print(f"Translation issues for {lang}:")
        for issue in issues:
            print(f"  ❌ {issue}")
    else:
        print(f"✓ {lang} translation valid")
```


## i18n Best Practices

### Do's

- ✓ Use translation keys (never hardcode strings)
- ✓ Provide context for translators in comments
- ✓ Extract strings to translation files
- ✓ Test with different string lengths
- ✓ Use professional translators for tier-1 languages
- ✓ Validate translations regularly
- ✓ Support RTL languages (Arabic, Hebrew)

### Don'ts

- ✗ Hardcode strings in UI
- ✗ Assume all languages read left-to-right
- ✗ Ignore character encoding
- ✗ Skip cultural sensitivity checks
- ✗ Use Google Translate for production
- ✗ Mix translation keys and untranslated strings
- ✗ Forget to test with actual native speakers

### Quality Checklist

- [ ] All strings extracted to translation files
- [ ] Fallback to English configured
- [ ] Language selector visible in UI
- [ ] Translations professionally reviewed
- [ ] RTL languages supported
- [ ] Date/time formats regional
- [ ] Currency display localized
- [ ] Test with different screen sizes per language
- [ ] Performance: lazy-load large translation files
- [ ] Analytics: track language preferences
